In [3]:
# --- Imports ---

from dotenv import load_dotenv
load_dotenv()

import pandas as pd
import numpy as np
from pathlib import Path

In [4]:
# --- Constants ---

PROJECT_ROOT = Path.cwd().parent
INTERIM_DIR  = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(exist_ok=True)

In [5]:
# --- Data fetching report ---

report = pd.read_csv(INTERIM_DIR / "collection_report.csv")

In [6]:
report

,dataset,code,level,nuts0,nuts1,nuts2,year_min,year_max,n_years,shape,missing_pct,notes
0,aerobic_activity,hlth_ehis_pe2e,all,32,0,0,2014,2019,2,"(32, 2)",7.8,% meeting 150+ min/week. 2014 and 2019 only.
1,family_contact,ilc_scp09,all,36,0,0,2015,2022,2,"(36, 2)",8.3,% getting together with family at least weekly...
2,gdp_per_capita,nama_10r_2gdp,all,33,110,285,2000,2024,25,"(428, 25)",3.1,GDP per capita in PPS. NUTS0-2 confirmed.
3,heavy_drinking,hlth_ehis_al3e,all,32,0,0,2014,2019,2,"(32, 2)",7.8,% drinking heavily at least once/week. 2014 an...
4,internet_usage,isoc_r_iuse_i,all,38,126,225,2006,2025,20,"(389, 20)",36.2,% regularly using internet. NUTS2 confirmed.
5,life_expectancy,demo_r_mlifexp,all,37,125,345,1990,2024,35,"(507, 35)",24.3,Life expectancy at birth. No total sex — use F...
6,life_satisfaction,ilc_pw01,all,37,0,0,2013,2025,7,"(37, 7)",10.0,Meaning of life
7,obesity_rate,sdg_02_10,all,37,0,0,2008,2025,6,"(37, 6)",19.8,% population BMI>=30.
8,poverty_risk,ilc_li41,all,38,100,283,2003,2025,23,"(421, 23)",44.8,No sub-dimensions — just filter unit. NUTS2 co...
9,smoking,hlth_ehis_sk3e,all,32,0,0,2014,2019,2,"(32, 2)",3.1,% smokers total (light + heavy). 2014 and 2019...


In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
INTERIM_DIR  = PROJECT_ROOT / "data" / "interim"

# Datasets we care about for PCA + validation
KEY_DATASETS = [
    "life_expectancy",
    "unemployment_rate",
    "poverty_risk",
    "internet_usage",
    "social_trust",
    "life_satisfaction",
    "gdp_per_capita",
    "family_contact",
    "heavy_drinking",
    "fruit_veggies",
    "obesity_rate",
    "smoking",
    "social_contact",
    "social_expenditure",
    "social_support"
]

with open(PROJECT_ROOT / "data" / "interim" / "coverage_report.txt", "w") as f:
    for name in KEY_DATASETS:
        df = pd.read_parquet(INTERIM_DIR / f"{name}.parquet")
        available = df.notna().sum()
        total = len(df)
        f.write(f"\n── {name} (total regions: {total}) ──\n")
        f.write(available.to_string())
        f.write("\n")

print("Saved — open data/interim/coverage_report.txt")

Saved — open data/interim/coverage_report.txt


In [12]:
ALL_DATASETS = [
    "life_expectancy", "unemployment_rate", "poverty_risk",
    "internet_usage", "gdp_per_capita", "social_trust",
    "life_satisfaction", "aerobic_activity", "family_contact",
    "heavy_drinking", "obesity_rate", "smoking", "social_contact",
    "social_expenditure", "social_support",
]

THRESHOLD_YEARS = [2005, 2008, 2011, 2013, 2014, 2015, 2016, 2017, 2018, 2019 ,2020, 2021, 2022, 2023, 2024, 2025]

rows = {}
totals = {}
for name in ALL_DATASETS:
    path = INTERIM_DIR / f"{name}.parquet"
    if not path.exists():
        continue
    df = pd.read_parquet(path)
    totals[name] = len(df)
    row = {}
    for year in THRESHOLD_YEARS:
        if year in df.columns:
            row[year] = int(df[year].notna().sum())
        else:
            row[year] = "-"
    rows[name] = row

overview = pd.DataFrame(rows).T
overview.columns = THRESHOLD_YEARS
overview["total"] = overview.index.map(lambda n: f"(of {totals[n]})")
print(overview.to_string())

                   2005 2008 2011 2013 2014 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024 2025     total
life_expectancy     410  418  458  471  494  494  495  503  503  453  409  405  399  435  402    -  (of 507)
unemployment_rate   413  464  469  484  484  484  485  485  485  486  420  428  425  427  429  429  (of 510)
poverty_risk        136  187  196  209  217  217  233  239  264  278  274  353  395  392  389  372  (of 421)
internet_usage        -   32  263  284  328  314  322  337  334  349  298  334  319  347  343  352  (of 389)
gdp_per_capita      399  415  415  427  428  428  428  428  428  428  428  428  420  415  415    -  (of 428)
social_trust          -    -    -   34    -    -    -    -   37    -    -   32   33   34   32   30   (of 37)
life_satisfaction     -    -    -   34    -    -    -    -   37    -    -   32   34   34   32   30   (of 37)
aerobic_activity      -    -    -    -   29    -    -    -    -   30    -    -    -    -    -    -   (of 32)
family_contact     